# Event Weights

## Problem Definition

**Question.** How much independent price information does each observed overlapping event contribute?

**Role in the workflow.** Create deterministic sample weights for primary/meta fitting without generating synthetic events.

**Inputs.** The labeled event Parquet and local dollar-bar close path.

**Outputs.** `data/research_data/events/aapl_news_modeling_weighted_2025-01-01_2025-12-31.parquet`.

**Why this method.** Concurrency, average uniqueness, return attribution, and time decay follow AFML sample-weight logic on actual event intervals.

**Assumptions.** Weights affect fitting only; labels and returns are unchanged, and every value is computed from its recorded event information set.

**Handoff.** The weighted event table to the four Method References and primary/meta models.


## Real Data and Weight Construction

No random horizons or synthetic trades are used. The later holdout split is chronological; these deterministic weights do not select a model or expose holdout performance.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing.event_weights import (
    apply_time_decay,
    compute_average_uniqueness_weights,
    compute_return_attribution_weights,
    count_concurrent_events,
)

period = "2025-01-01_2025-12-31"
event_path = PROJECT_ROOT / f"data/research_data/events/aapl_news_primary_model_{period}.parquet"
dollar_path = PROJECT_ROOT / f"data/research_data/market/features/aapl_dollar_bar_{period}.parquet"

events = pd.read_parquet(event_path).sort_values("event_start", ignore_index=True)
dollar_bars = pd.read_parquet(dollar_path).sort_values("end").drop_duplicates("end", keep="last")
events["event_start"] = pd.to_datetime(events["event_start"], utc=True)
events["event_end"] = pd.to_datetime(events["event_end"], utc=True)

information_sets = events.set_index("event_start")["event_end"]
close = dollar_bars.set_index("end")["close"].astype(float)
concurrency = count_concurrent_events(close.index, information_sets, information_sets.index)
uniqueness = compute_average_uniqueness_weights(information_sets, concurrency, information_sets.index)
return_attribution = compute_return_attribution_weights(information_sets, concurrency, close, information_sets.index)

base_weight = return_attribution.clip(lower=return_attribution[return_attribution > 0].min())
time_decay = apply_time_decay(base_weight, clf_last_w=0.50)
sample_weight = base_weight * time_decay
sample_weight *= len(sample_weight) / sample_weight.sum()

weight_table = pd.DataFrame(
    {
        "average_uniqueness_weight": uniqueness,
        "return_attribution_weight": return_attribution,
        "time_decay_weight": time_decay,
        "sample_weight": sample_weight,
    }
).rename_axis("event_start").reset_index()

weighted_events = events.merge(weight_table, on="event_start", how="left", validate="one_to_one")
assert weighted_events["return_attribution_weight"].ge(0).all()
assert weighted_events[["average_uniqueness_weight", "time_decay_weight", "sample_weight"]].gt(0).all().all()

weighted_path = event_path.with_name(f"aapl_news_modeling_weighted_{period}.parquet")
weighted_events.to_parquet(weighted_path, index=False)

display(weight_table.describe().T)
print(weighted_path)


## Results, Limitations, and Handoff

Return-attribution weights can concentrate influence in volatile events, while time decay downweights older observations. The notebook reports these distributions and does not claim that weighting guarantees better generalization.

The next notebook receives the observed weighted-event Parquet. No conclusion in this notebook is evidence of live-trading profitability.
